# 02 · Spatial folds (V3's blockCV folds)

V3 made the block size and the folds in R with blockCV 4.0-1:
* `cv_spatial_autocor` on every 1.5 m depth slice (the Clay indicator) gave one range per slice;
* the **block size = median of the slice ranges = 106 m**;
* `cv_spatial` made 5 folds of 106 m square blocks, and of 212 m and 318 m (2× and 3×) for the sensitivity runs.

R is **not** run again here: the PC now has blockCV 4.0-2, which could give other folds. V3's fold files are read as they are.

**Run time:** a few seconds.

In [ ]:
import json, sys, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "settings.toml").exists())))
import helpers as H

S = H.load_settings()
T0 = time.time()
print("QUICK TEST RUN - not thesis results" if S["run"]["quick"] else "FULL RUN", "| V3 folder:", S["v3_root"])
dec = json.loads((S["v3_root"] / "data" / "folds" / "block_size_decision.json").read_text())
print("Method:", dec["method"])
print("blockCV", dec.get("blockcv_version"), "| automap", dec.get("automap_version"))
ranges = pd.Series(dec["ranges_m"], name="range_m")
display(ranges.to_frame())
print(f"Median of {len(ranges)} slices = {ranges.median():.1f} m -> block size {dec['block_size_m']} m")
print("Block sizes used:", H.block_sizes(S))

In [ ]:
cells = pd.read_csv(H.processed_dir(S) / "cells.csv")
dev = cells[cells.set == "development"]
rows = []
for b in H.block_sizes(S):
    f = H.load_folds(S, b)
    assert set(f.index[f > 0]) == set(dev.BH_Code), "fold file and development boreholes differ"   # fold 0 = 5 km ring (not used in V3)
    d = dev.assign(fold=dev.BH_Code.map(f))
    t = d.groupby("fold").agg(boreholes=("BH_Code", "nunique"), cells=("y", "size"), clay_share=("y", "mean"))
    rows.append(t.assign(block_m=b).reset_index())
folds_table = pd.concat(rows)
folds_table.to_csv(H.out_dir(S, "02_folds") / "folds_summary.csv", index=False)
display(folds_table.pivot_table(index="fold", columns="block_m", values=["boreholes", "clay_share"]).round(3))
print(f"Finished in {time.time() - T0:.0f} s. Next: notebook 03.")